# 11 — Virtual Screening 性能評価チュートリアル
# Virtual Screening Evaluation Tutorial

**目的**: ドッキングスコアを用いた仮想スクリーニング（VS）の定量的評価  
**使用指標**: Enrichment Factor (EF) / Normalised EF / ROC-AUC / BEDROC  
**ユースケース**: AlphaFold2 等で生成した複数コンフォメーションの VS 性能比較

---

## 評価指標の概要

| 指標 | 意味 | 理想値 | 参考 |
|------|------|--------|------|
| **EF(x%)** | ライブラリ上位 x% に占めるアクティブ割合 ÷ ランダム期待値 | 最大値 (EF_max) | Jain & Nicholls 2008 |
| **NEF(x%)** | EF / EF_max (0–1 に正規化) | 1.0 | |
| **ROC-AUC** | 全アクティブの平均順位性能 | 1.0 | |
| **BEDROC** | 上位回収に指数重みを付けた敏感指標 (α=20) | 1.0 | Truchon & Bayly 2007 |

> **BEDROC の注意**: ランダム基準値は 0.5 **ではなく**、Ra(アクティブ割合) と α に依存します。  
> Ra=0.1, α=20 の場合、ランダム基準 ≈ 0.116。絶対値ではなく相対比較に使用してください。

---

## このノートブックで学べること

1. アクティブ/デコイ ラベルを付けた DataFrame の準備方法
2. `VirtualScreeningEvaluator` で単一・複数コンフォメーションを一括評価
3. ROC カーブ・エンリッチメントカーブ・EF 棒グラフの可視化
4. `plot_vs_summary_table()` によるサマリーレポート生成

## 1. CONFIG セル

データセットを変える場合はここだけ書き換えてください。

In [ ]:
from pathlib import Path

# ---- Paths ----
DATA_DIR    = Path("../data")           # CSV or SDF with scores + labels
RESULTS_DIR = Path("../results/11_vs")  # output figures
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Column names in your CSV ----
SCORE_COL       = "docking_score"   # float, lower = better
LABEL_COL       = "active"          # int,   1 = active / 0 = decoy
GROUP_COL       = "conformation"    # optional: compare multiple conformations

# ---- Evaluator settings ----
HIGHER_IS_BETTER = False            # docking score convention
ALPHA            = 20.0             # BEDROC alpha

## 2. サンプルデータの生成 / Generate Sample Data

独自のデータがある場合はこのセルをスキップして CSV を読み込んでください。

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

N_TOTAL   = 1000   # total compounds
N_ACTIVES = 50     # known actives
N_CONFS   = 3      # number of receptor conformations

rows = []
for conf_idx in range(1, N_CONFS + 1):
    conf_name = f"conf_{conf_idx:02d}"
    
    # Actives: biased towards better (lower) scores
    active_scores = rng.normal(loc=-8.0 + conf_idx * 0.3, scale=0.8, size=N_ACTIVES)
    # Decoys: random
    decoy_scores  = rng.normal(loc=-5.0, scale=1.5, size=N_TOTAL - N_ACTIVES)
    
    scores = np.concatenate([active_scores, decoy_scores])
    labels = np.array([1] * N_ACTIVES + [0] * (N_TOTAL - N_ACTIVES))
    
    for s, l in zip(scores, labels):
        rows.append({SCORE_COL: s, LABEL_COL: l, GROUP_COL: conf_name})

df = pd.DataFrame(rows)
print(f"Dataset: {len(df)} rows, {df[GROUP_COL].nunique()} conformations")
print(f"Actives per conformation: {df.groupby(GROUP_COL)[LABEL_COL].sum().to_dict()}")
df.head()

## 3. 評価の実行 / Run Evaluation

In [ ]:
from docking_analysis.analysis.vs_metrics import VirtualScreeningEvaluator

ev = VirtualScreeningEvaluator(alpha=ALPHA)

# --- 複数コンフォメーションを一括評価 ---
results = ev.run_from_df(
    df,
    score_col=SCORE_COL,
    label_col=LABEL_COL,
    group_col=GROUP_COL,
    higher_is_better=HIGHER_IS_BETTER,
)

print(f"Evaluated {len(results)} conformations")
for name, res in sorted(results.items()):
    print(f"  {name}: EF1%={res.ef[1.0]:.2f}, EF5%={res.ef[5.0]:.2f}, "
          f"ROC-AUC={res.roc_auc:.3f}, BEDROC={res.bedroc:.3f}")

## 4. サマリーテーブル / Summary Table

In [ ]:
from docking_analysis.visualization.vs_plots import plot_vs_summary_table

summary_df = plot_vs_summary_table(results, ef_percentiles=(1.0, 5.0, 10.0))
display(summary_df)

## 5. ROC カーブの可視化 / ROC Curves

In [ ]:
import matplotlib.pyplot as plt
from docking_analysis.visualization.vs_plots import plot_roc_curves

fig = plot_roc_curves(
    results,
    output_path=RESULTS_DIR / "roc_curves.png",
    title="ROC Curves — Receptor Conformation Comparison",
)
plt.show()

## 6. エンリッチメントカーブ / Enrichment Curves

In [ ]:
from docking_analysis.visualization.vs_plots import plot_enrichment_curves

fig = plot_enrichment_curves(
    results,
    output_path=RESULTS_DIR / "enrichment_curves.png",
    title="Enrichment Curves — % Library vs % Actives Recovered",
)
plt.show()

## 7. EF 棒グラフ / EF Bar Chart

In [ ]:
from docking_analysis.visualization.vs_plots import plot_ef_bars

fig = plot_ef_bars(
    results,
    pct=5.0,
    normalised=True,   # NEF (0–1 scale)
    output_path=RESULTS_DIR / "nef5_bars.png",
    title="Normalised EF5% by Conformation",
)
plt.show()

## 8. 上位コンフォメーションの詳細確認 / Best Conformation Deep Dive

サマリーテーブルの最上位コンフォメーションについて詳細を確認します。

In [ ]:
# summary_df はすでに EF5% 降順ソート済み
best_conf = summary_df.index[0]
best = results[best_conf]

print(f"Best conformation: {best_conf}")
print(f"  N_total  : {best.n_total}")
print(f"  N_actives: {best.n_actives} (Ra = {best.n_actives/best.n_total:.1%})")
print()
print("EF values:")
for pct in sorted(best.ef):
    ef    = best.ef[pct]
    ef_mx = best.ef_max[pct]
    nef   = best.ef_norm[pct]
    print(f"  EF{pct:4.1f}%  = {ef:.3f}  (EF_max={ef_mx:.3f}, NEF={nef:.3f})")
print()
print(f"ROC-AUC : {best.roc_auc:.4f}")
print(f"BEDROC  : {best.bedroc:.4f}")

---

## 参考文献

- Truchon, J.-F. & Bayly, C.I. (2007) *J. Chem. Inf. Model.* **47**, 488–508 — BEDROC  
- Jain, A.N. & Nicholls, A. (2008) *J. Comput.-Aided Mol. Des.* **22**, 133–139 — EF の適切な使い方の注意点